# MCP Gateway & Registry Workshop

Deploy the [MCP Gateway & Registry](https://github.com/agentic-community/mcp-gateway-registry) on your EKS cluster. This platform centralizes access to MCP servers and AI agents with authentication, governance, and dynamic tool discovery.

| Module | Description | Dependency |
|--------|-------------|------------|
| **Module 0** | Clone the upstream chart and build dependencies | |
| **Module 1** | Deploy the full stack (MongoDB, Keycloak, Registry, Auth Server) | Module 0 |
| **Module 2** | Access the Registry UI and retrieve credentials | Module 1 |
| **Module 3** | Build and deploy a sample MCP server | Module 1 |
| **Module 4** | Register the server and enable it for routing | Module 3 |
| **Module 5** | Call a tool through the gateway, and see registry-controlled access | Module 4 |


## Prerequisites

* This notebook **must be run** on the ML Ops desktop created via [quick start setup](../../../README.md#quick-start-basic). This notebook can also be used on an ML Ops desktop created via [advanced setup](../../../README.md#advanced-setup), but the user will need to appropriately adapt the notebook for use with the advanced setup.
* `helm` CLI v3.0+ installed.
* `kubectl` configured to access your EKS cluster.

## Architecture

The stack deploys the following components into the `mcp-gateway` namespace:

```
┌──────────────────────────────────────────────────────────────────────┐
│                           EKS Cluster                               │
│                                                                      │
│  ┌────────────────────────────────────────────────────────────────┐  │
│  │                    mcp-gateway namespace                       │  │
│  │                                                                │  │
│  │   ┌───────────────┐                                            │  │
│  │   │ Nginx Gateway │─── /registry/*  ──► Registry (8000)       │  │
│  │   │   (8080)      │─── /auth-server/*► Auth Server (8888)     │  │
│  │   │               │─── /keycloak/*  ──► Keycloak (80)         │  │
│  │   │               │─── /mcpgw/*     ──► MCPGW (8003)         │  │
│  │   └───────────────┘                                            │  │
│  │                                                                │  │
│  │   ┌─────────────┐    ┌──────────────┐    ┌────────────────┐   │  │
│  │   │  Keycloak   │    │  Auth Server  │    │   Registry     │   │  │
│  │   │   (IdP)     │◄──►│  (OAuth)      │◄──►│   (Web UI)     │   │  │
│  │   └──────┬──────┘    └──────────────┘    └────────────────┘   │  │
│  │          │                                                     │  │
│  │   ┌──────▼──────┐    ┌──────────────┐                         │  │
│  │   │ PostgreSQL  │    │   MongoDB     │                         │  │
│  │   │ (Keycloak)  │    │  (Registry)   │                         │  │
│  │   └─────────────┘    └──────────────┘                         │  │
│  │                                                                │  │
│  │   ┌──────────────┐                                             │  │
│  │   │   MCPGW      │                                             │  │
│  │   │ (MCP Server) │                                             │  │
│  │   └──────────────┘                                             │  │
│  └────────────────────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────────────────────┘
         │
         ▼
   port-forward
   :8080 (Gateway)
```

---
## Module 0: Clone Chart and Build Dependencies

The MCP Gateway Registry Helm chart uses local file references for its sub-charts, so we need to clone the upstream repository and build dependencies from source.

### Step 0.1: Set Up Paths

In [ ]:
import os

REPO_DIR = os.path.expanduser("~/amazon-eks-machine-learning-with-terraform-and-kubeflow")
EXAMPLE_DIR = os.path.join(REPO_DIR, "examples/agentic/mcp-gateway-registry")
UPSTREAM_DIR = os.path.expanduser("~/mcp-gateway-registry")
CHART_DIR = os.path.join(UPSTREAM_DIR, "charts/mcp-gateway-registry-stack")

os.environ["REPO_DIR"] = REPO_DIR
os.environ["EXAMPLE_DIR"] = EXAMPLE_DIR
os.environ["UPSTREAM_DIR"] = UPSTREAM_DIR
os.environ["CHART_DIR"] = CHART_DIR

NAMESPACE = "mcp-gateway"
os.environ["NAMESPACE"] = NAMESPACE

print(f"Example Dir:  {EXAMPLE_DIR}")
print(f"Upstream Dir: {UPSTREAM_DIR}")
print(f"Chart Dir:    {CHART_DIR}")
print(f"Namespace:    {NAMESPACE}")

### Step 0.2: Clone the Upstream Repository

In [ ]:
%%bash
RELEASE_TAG="v1.0.16"

if [  ! -d "$UPSTREAM_DIR" ]; then
  # Use --depth 1 if you only need the files at this tag (faster)
  git clone --branch "$RELEASE_TAG" --depth 1 https://github.com/agentic-community/mcp-gateway-registry.git "$UPSTREAM_DIR"
else
  cd "$UPSTREAM_DIR"
  # --tags ensures the specific tag is downloaded even if it's new
  git fetch origin --tags
  git checkout "$RELEASE_TAG"
fi


### Step 0.3: Add Nginx Gateway Template

Copy the nginx gateway template into the cloned chart. This adds an in-cluster nginx reverse proxy that provides a single entry point for all services via path-based routing, so only one `kubectl port-forward` is needed.

In [ ]:
%%bash
cp $EXAMPLE_DIR/templates/nginx-gateway.yaml $CHART_DIR/templates/
echo "Copied nginx-gateway.yaml to chart templates"

### Step 0.4: Build Helm Dependencies

In [ ]:
%%bash
cd $CHART_DIR
helm dependency build
helm dependency update
echo "Dependencies built successfully"

---
## Module 1: Deploy the Full Stack

Install the MCP Gateway Registry stack using the workshop values file. This deploys MongoDB, Keycloak, the Registry, Auth Server, and MCPGW into the `mcp-gateway` namespace.

All ingress is disabled — we use `kubectl port-forward` to access services.

### Step 1.1: Generate a MongoDB Password

In [ ]:
import secrets
import string

MONGO_USERNAME = 'mcpgateway'
MONGO_PASSWORD = ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(32))
os.environ["MONGO_USERNAME"] = MONGO_USERNAME
os.environ["MONGO_PASSWORD"] = MONGO_PASSWORD
print("MongoDB credentials generated (stored in environment)")

### Step 1.2: Install the Helm Chart

This takes **5-10 minutes**. Keycloak and MongoDB need to initialize before dependent services start.

In [ ]:
%%bash
cd $CHART_DIR
helm install mcp-gateway-registry \
  -n $NAMESPACE --create-namespace \
  -f $EXAMPLE_DIR/values.yaml \
  --set mongodb.password="$MONGO_PASSWORD" \
  --set mongodb-configure.mongodb.password="$MONGO_PASSWORD" \
  --set mongodb-configure.mongodb.username="$MONGO_USERNAME" \
  --set mongodb.user="$MONGO_USERNAME" \
  .

### Step 1.3: Patch Registry Admin Password

The upstream chart has a bug where the `keycloak-configure` job generates the admin password and saves it to `registry-login-credentials`, but never wires it into the registry deployment. We wait for the job to complete, then patch `registry-secret` with the `ADMIN_PASSWORD` the container expects.

In [ ]:
%%bash
echo "Waiting for keycloak-configure job to complete..."
kubectl wait --for=condition=complete job/setup-keycloak -n $NAMESPACE --timeout=600s

echo "Patching registry-secret with ADMIN_PASSWORD..."
ADMIN_PASS=$(kubectl get secret registry-login-credentials -n $NAMESPACE -o jsonpath='{.data.REGISTRY_ADMIN_PASSWORD}' | base64 -d)
kubectl patch secret registry-secret -n $NAMESPACE -p '{"data":{"ADMIN_PASSWORD":"'$(echo -n "$ADMIN_PASS" | base64)'"}}'

echo "Restarting registry deployment..."
kubectl rollout restart deployment registry -n $NAMESPACE
echo "Done."

### Step 1.4: Wait for Pods to Be Ready

Monitor the rollout. All pods should now come up successfully.

In [ ]:
%%bash
echo "Waiting for deployments in $NAMESPACE namespace..."
for deploy in $(kubectl get deployments -n $NAMESPACE -o jsonpath='{.items[*].metadata.name}'); do
  echo "Waiting for deployment/$deploy..."
  kubectl rollout status deployment/$deploy -n $NAMESPACE --timeout=600s
done
echo ""
echo "Waiting for statefulsets in $NAMESPACE namespace..."
for sts in $(kubectl get statefulsets -n $NAMESPACE -o jsonpath='{.items[*].metadata.name}'); do
  echo "Waiting for statefulset/$sts..."
  kubectl rollout status statefulset/$sts -n $NAMESPACE --timeout=600s
done
echo ""
kubectl get pods -n $NAMESPACE

---
## Module 2: Access the Registry UI

With all services running, use port-forwarding to access the Registry web UI.

### Step 2.1: Retrieve Keycloak Credentials

The `keycloak-configure` job creates a user for the Registry. Retrieve the credentials from the job logs.

In [ ]:
%%bash
echo "Looking for completed keycloak-configure job..."
COMPLETED_POD=$(kubectl get pods -n $NAMESPACE -l job-name=setup-keycloak \
  --field-selector=status.phase==Succeeded -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$COMPLETED_POD" ]; then
  echo "Keycloak configure job has not completed yet. Re-run this cell after pods are ready."
else
  echo "Credentials from keycloak-configure job:"
  echo "=========================================="
  kubectl logs -n $NAMESPACE $COMPLETED_POD --tail=20
fi

### Step 2.2: Port-Forward the Gateway

An nginx gateway pod runs inside the cluster and routes all traffic to the backend services by path. You only need a single port-forward.

Run the following command in a **separate terminal** on the ML Ops desktop, then open the UI in your browser.

In [ ]:
%%bash
echo "Run the following command in a separate terminal on the ML Ops desktop:"
echo ""
echo "  kubectl port-forward svc/nginx-gateway -n $NAMESPACE 8080:8080"
echo ""
echo "Then open: http://localhost:8080/registry/"
echo ""
echo "Log in with the credentials from Step 2.1."

### Step 2.3: Explore the Registry

Once logged in, you can:

- **Register MCP Servers** — Add your MCP servers (e.g., EKS MCP Server, custom tools) to the registry for centralized access and governance.
- **Register A2A Agents** — Register AI agents that support the Agent-to-Agent protocol for discovery and communication.
- **Configure Access Control** — Set up groups and fine-grained permissions for users and service accounts via the IAM Settings page.
- **Search & Discover** — Use semantic search to find registered servers, tools, and agents by natural language description.

For full documentation on configuring the registry, see the [MCP Gateway & Registry docs](https://github.com/agentic-community/mcp-gateway-registry).

---
## Module 3: Deploy a Sample MCP Server

The registry does not *host* MCP servers — it routes to them. To test end-to-end routing we need a real MCP server running in the cluster. This module builds a minimal **time** server (one tool, `get_current_time`) and deploys it into the `mcp-gateway` namespace.

The server speaks **streamable-HTTP** at `/mcp` — the transport the gateway proxies and the registry health-checks.

> **Note:** Routing only works when the registry runs in `with-gateway` mode. The workshop `values.yaml` sets `registry.app.deploymentMode: with-gateway`, so a fresh install is already configured for this. In `registry-only` mode the registry is a catalog and never generates routing.

### Step 3.1: Build, Push, and Deploy

In [ ]:
%%bash
cd $EXAMPLE_DIR/sample-time-server
chmod +x build-and-deploy.sh
./build-and-deploy.sh

### Step 3.2: Verify the Server Speaks MCP

Send an MCP `initialize` from inside the cluster. A healthy server returns HTTP 200 and an `mcp-session-id` header — exactly what the registry's health check looks for.

In [ ]:
%%bash
kubectl run mcp-test --rm -i --restart=Never -n $NAMESPACE --image=curlimages/curl -- \
  curl -sS -D - -o /dev/null \
  -X POST http://sample-time-server.$NAMESPACE.svc.cluster.local:8000/mcp \
  -H 'Content-Type: application/json' \
  -H 'Accept: application/json, text/event-stream' \
  -d '{"jsonrpc":"2.0","id":"1","method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"test","version":"1"}}}'

---
## Module 4: Register the Server

Register the time server with the registry and enable it. On enable, the registry health-checks the backend and — in `with-gateway` mode — auto-generates an nginx `location` block (protected by `auth_request`) that proxies to the server.

> **Two trailing-slash gotchas** (both required, learned the hard way):
> 1. `path` must end with `/` (e.g. `/sample-time/`). The toggle endpoint looks up the path *with* a trailing slash; registering without one makes toggle fail.
> 2. `proxy_pass_url` must end with `/`. The registry copies it verbatim into nginx `proxy_pass`; with a trailing slash nginx strips the `/registry/<path>/` prefix and forwards `/mcp` to the backend. Without it, the backend receives the full path and returns 404.

These cells assume the gateway port-forward from **Step 2.2** is running (`kubectl port-forward svc/nginx-gateway -n mcp-gateway 8080:8080`).

### Step 4.1: Log In and Register

In [ ]:
import subprocess, base64, requests

GATEWAY  = "http://localhost:8080"
REGISTRY = f"{GATEWAY}/registry"
SERVER_PATH = "/sample-time/"   # trailing slash required
PROXY_URL   = f"http://sample-time-server.{NAMESPACE}.svc.cluster.local:8000/"  # trailing slash required

def get_secret(name, key):
    raw = subprocess.check_output(
        ["kubectl", "get", "secret", name, "-n", NAMESPACE,
         "-o", "jsonpath={.data." + key + "}"]
    )
    return base64.b64decode(raw).decode()

admin_user = get_secret("registry-login-credentials", "REGISTRY_ADMIN_NAME")
admin_pass = get_secret("registry-login-credentials", "REGISTRY_ADMIN_PASSWORD")

session = requests.Session()
r = session.post(f"{REGISTRY}/api/auth/login",
                 data={"username": admin_user, "password": admin_pass},
                 allow_redirects=False)
print("login:", r.status_code,
      "| session cookie set:", "mcp_gateway_session" in session.cookies.get_dict())

# Idempotent: remove any prior registration so re-runs don't 409
for p in (SERVER_PATH.rstrip("/"), SERVER_PATH):
    session.post(f"{REGISTRY}/api/servers/remove", data={"path": p})

r = session.post(f"{REGISTRY}/api/register", data={
    "name": "Sample Time",
    "description": "Minimal MCP time server for testing",
    "path": SERVER_PATH,
    "proxy_pass_url": PROXY_URL,
    "num_tools": 2,
    "supported_transports": "streamable-http",
})
print("register:", r.status_code, r.json().get("message"))

### Step 4.2: Enable It

Toggling the server on triggers the health check and nginx config regeneration. A `healthy` status with `num_tools` populated means the registry reached the server and discovered its tools.

In [ ]:
r = session.post(f"{REGISTRY}/api/toggle/sample-time/", data={"enabled": "on"})
print("toggle:", r.status_code)
print(r.json())

---
## Module 5: Call a Tool Through the Gateway

Now call the server's tool through the gateway. Requests are authenticated with a machine-to-machine (M2M) JWT minted from Keycloak — the `auth_request` on the generated nginx block validates it before proxying to the server.

### Step 5.1: Mint a Token and Call `get_current_time`

In [ ]:
import json

CLIENT_ID     = "mcp-gateway-m2m"
CLIENT_SECRET = get_secret("keycloak-client-secret", "KEYCLOAK_M2M_CLIENT_SECRET")
TOKEN_URL = f"{GATEWAY}/keycloak/realms/mcp-gateway/protocol/openid-connect/token"
MCP_URL   = f"{GATEWAY}/registry/sample-time/mcp"

def mint_token():
    r = requests.post(TOKEN_URL, data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    })
    r.raise_for_status()
    return r.json()["access_token"]

def mcp_headers(token, session_id=None):
    h = {"Authorization": f"Bearer {token}",
         "Content-Type": "application/json",
         "Accept": "application/json, text/event-stream"}
    if session_id:
        h["Mcp-Session-Id"] = session_id
    return h

def parse_sse(text):
    for line in text.splitlines():
        if line.startswith("data:"):
            return json.loads(line[5:].strip())
    return {"raw": text}

def call_tool(token, name, arguments):
    # 1. initialize -> capture session id
    r = requests.post(MCP_URL, headers=mcp_headers(token), json={
        "jsonrpc": "2.0", "id": "1", "method": "initialize",
        "params": {"protocolVersion": "2024-11-05", "capabilities": {},
                   "clientInfo": {"name": "notebook", "version": "1"}}})
    sid = r.headers.get("mcp-session-id")
    if not sid:
        return f"BLOCKED (initialize HTTP {r.status_code})"
    # 2. initialized notification
    requests.post(MCP_URL, headers=mcp_headers(token, sid),
                  json={"jsonrpc": "2.0", "method": "notifications/initialized"})
    # 3. tools/call
    r = requests.post(MCP_URL, headers=mcp_headers(token, sid), json={
        "jsonrpc": "2.0", "id": "2", "method": "tools/call",
        "params": {"name": name, "arguments": arguments}})
    res = parse_sse(r.text)
    try:
        return res["result"]["content"][0]["text"]
    except Exception:
        return f"HTTP {r.status_code}: {r.text[:160]}"

token = mint_token()
print("token length:", len(token))
print("get_current_time(America/New_York) ->",
      call_tool(token, "get_current_time", {"tz": "America/New_York"}))

### Step 5.2: Registry-Controlled Access (the payoff)

The whole point of the gateway: **registry state controls live traffic, with no redeploy.** Disable the server and the same call is blocked (its nginx route disappears); re-enable it and the call works again.

In [ ]:
import time

print("Disabling the server in the registry...")
session.post(f"{REGISTRY}/api/toggle/sample-time/", data={"enabled": "off"})
time.sleep(2)
print("  call result:", call_tool(mint_token(), "get_current_time", {"tz": "UTC"}))

print("\nRe-enabling the server...")
session.post(f"{REGISTRY}/api/toggle/sample-time/", data={"enabled": "on"})
time.sleep(2)
print("  call result:", call_tool(mint_token(), "get_current_time", {"tz": "UTC"}))

---
## Troubleshooting

| Issue | Cause | Solution |
|-------|-------|----------|
| Keycloak pod stuck in `CrashLoopBackOff` | PostgreSQL not ready | Wait for the PostgreSQL pod to be `Running` first |
| Auth server not starting | `keycloak-configure` job not completed | Check job status: `kubectl get jobs -n mcp-gateway` |
| Cannot log in to Registry UI | Port-forward not running | Ensure `kubectl port-forward svc/nginx-gateway -n mcp-gateway 8080:8080` is active |
| MongoDB pod pending | No storage class available | Verify a default StorageClass exists: `kubectl get sc` |
| `toggle` returns HTTP 500 / "not found" | Server registered with a `path` missing its trailing slash | Register with `path=/sample-time/` (trailing slash) |
| Tool call returns 404 from the backend | `proxy_pass_url` registered without a trailing slash | Register with `proxy_pass_url=http://...:8000/` (trailing slash) |
| Tool call returns 405 / route missing | Server is disabled in the registry, or running in `registry-only` mode | Enable it (Step 4.2); confirm `deploymentMode: with-gateway` |


In [ ]:
# Useful troubleshooting commands
!kubectl get pods -n mcp-gateway
!kubectl get jobs -n mcp-gateway
!kubectl logs -n mcp-gateway -l app.kubernetes.io/name=registry --tail=30

---
## Cleanup

Uninstall the stack and delete the namespace.

In [ ]:
%%bash
helm uninstall mcp-gateway-registry -n $NAMESPACE
kubectl delete namespace $NAMESPACE
echo "Cleanup complete"